# Vybe — Exploratory Data Analysis
NYC neighborhood condition signals across 311, crime, crashes, traffic, subway, and social media.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.figsize'] = (10, 5)
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False

# load cleaned datasets (assumes data pipeline has already run)
from data.ingest import pull_311, pull_crime, pull_crashes, pull_traffic, pull_subway
from data.clean import clean_311, clean_crime, clean_crashes, clean_traffic, clean_subway

data_311       = clean_311(pull_311())
data_crime     = clean_crime(pull_crime())
data_crashes   = clean_crashes(pull_crashes())
data_traffic   = clean_traffic(pull_traffic())
data_subway    = clean_subway(pull_subway())

## 1. Dataset Overview

In [ ]:
datasets = {
    '311 Complaints': data_311,
    'Crime':          data_crime,
    'Crashes':        data_crashes,
    'Traffic':        data_traffic,
    'Subway':         data_subway
}

for name, df in datasets.items():
    null_pct = df.isnull().mean()
    top_nulls = null_pct[null_pct > 0].sort_values(ascending=False).head(5)
    print(f'\n {name}')
    print(f'shape: {df.shape}')
    if len(top_nulls):
        print(top_nulls.round(4).to_string())
    else:
        print('no nulls in key columns')

## 2. 311 Noise Complaints

In [ ]:
noise_311 = data_311[data_311['complaint_type'].str.contains('noise', case=False, na=False)].copy()
noise_311['hour'] = noise_311['created_date'].dt.hour

ax = noise_311.groupby('hour').size().plot(
    kind='bar', title='Noise Complaints by Hour of Day',
    color='#378ADD', edgecolor='none'
)
ax.set_xlabel('Hour')
ax.set_ylabel('Number of complaints')
plt.tight_layout()
plt.show()

print(f'\nTotal noise complaints: {len(noise_311):,}')
print(f'Peak hour: {noise_311.groupby("hour").size().idxmax()}:00')
print(f'\nTop complaint types:')
print(noise_311['complaint_type'].value_counts().head(10).to_string())

## 3. Crime Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# crime by category
data_crime['law_cat_cd'].value_counts().plot(
    kind='bar', ax=axes[0], title='Crime by Category',
    color='#D85A30', edgecolor='none'
)
axes[0].set_xlabel('')

# crime by borough
data_crime['boro_nm'].value_counts().head(5).plot(
    kind='bar', ax=axes[1], title='Crime by Borough',
    color='#D85A30', edgecolor='none'
)
axes[1].set_xlabel('')

plt.tight_layout()
plt.show()

print(f'\nTotal crime records: {len(data_crime):,}')
print(f'Date range: {data_crime["cmplnt_fr_dt"].min().date()} → {data_crime["cmplnt_fr_dt"].max().date()}')

## 4. Traffic Volume Patterns

In [ ]:
ax = data_traffic.groupby('hh')['vol'].sum().plot(
    kind='bar', title='Total Traffic Volume by Hour',
    color='#BA7517', edgecolor='none'
)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Total vehicle volume')
plt.tight_layout()
plt.show()

print('\nTop 10 busiest streets:')
print(data_traffic.groupby('street')['vol'].sum().sort_values(ascending=False).head(10).to_string())

## 5. Subway Ridership

In [ ]:
data_subway['hour'] = data_subway['transit_timestamp'].dt.hour

ax = data_subway.groupby('hour')['ridership'].sum().plot(
    kind='bar', title='Total Subway Ridership by Hour',
    color='#639922', edgecolor='none'
)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Total riders')
plt.tight_layout()
plt.show()

print('\nTop 10 busiest stations:')
print(
    data_subway.groupby('station_complex')['ridership']
    .sum().sort_values(ascending=False).head(10).to_string()
)

## 6. Crashes — Injury Analysis

In [ ]:
data_crashes['hour'] = data_crashes['crash_datetime'].dt.hour

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

data_crashes.groupby('hour').size().plot(
    kind='bar', ax=axes[0], title='Crashes by Hour',
    color='#A32D2D', edgecolor='none'
)
axes[0].set_xlabel('Hour')

injury_cols = [
    'number_of_persons_injured', 'number_of_pedestrians_injured',
    'number_of_cyclist_injured', 'number_of_motorist_injured'
]
injury_totals = {col.replace('number_of_', '').replace('_', ' '): data_crashes[col].sum()
                 for col in injury_cols if col in data_crashes.columns}

pd.Series(injury_totals).plot(
    kind='bar', ax=axes[1], title='Total Injuries by Type',
    color='#A32D2D', edgecolor='none'
)
axes[1].set_xlabel('')

plt.tight_layout()
plt.show()

print(f'\nTotal crashes: {len(data_crashes):,}')
print(f'Total persons injured: {data_crashes["number_of_persons_injured"].sum():,.0f}')

## 7. Hourly Activity Risk Score
Combined signal across noise, crashes, and traffic — weighted and normalized to 0–1.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

noise_311['hour'] = noise_311['created_date'].dt.hour
noise_hour   = noise_311.groupby('hour').size().reset_index(name='noise_count')
crashes_hour = data_crashes.groupby('hour').size().reset_index(name='crashes_count')
traffic_hour = data_traffic.groupby('hh')['vol'].sum().reset_index(name='traffic_count').rename(columns={'hh': 'hour'})

hourly = noise_hour.merge(crashes_hour, on='hour', how='outer').merge(traffic_hour, on='hour', how='outer').fillna(0)

scaler = MinMaxScaler()
hourly[['noise_scaled', 'crashes_scaled', 'traffic_scaled']] = scaler.fit_transform(
    hourly[['noise_count', 'crashes_count', 'traffic_count']]
)
hourly['risk_score'] = (
    0.333 * hourly['noise_scaled'] +
    0.333 * hourly['crashes_scaled'] +
    0.333 * hourly['traffic_scaled']
)

ax = hourly.set_index('hour')['risk_score'].sort_index().plot(
    kind='bar', title='Hourly Activity Risk Score (noise + crashes + traffic)',
    color='#7F77DD', edgecolor='none'
)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Risk score (0–1)')
plt.tight_layout()
plt.show()

print('\nTop 5 highest risk hours:')
print(hourly.sort_values('risk_score', ascending=False)[['hour', 'risk_score']].head().to_string(index=False))

## 8. Bluesky Social Signal Sample

In [ ]:
# assumes df_bluesky is loaded from the data gathering notebook
# from data.ingest import pull_bluesky
# from data.clean import clean_bluesky
# df_bluesky = clean_bluesky(pull_bluesky(username, password))

try:
    print('Bluesky posts sample:')
    print(df_bluesky[['query', 'text', 'engagement']].sample(10).to_string(index=False))
    print(f'\nTotal posts: {len(df_bluesky)}')
    print(f'Avg engagement: {df_bluesky["engagement"].mean():.1f}')
    print(f'\nTop queries by post count:')
    print(df_bluesky['query'].value_counts().to_string())
except NameError:
    print('df_bluesky not loaded — run the data gathering notebook first')

## 9. Key Insights

- **Noise peaks late night (10pm–1am)** — aligns with NYC nightlife patterns
- **Traffic peaks 4–5pm** — classic rush hour, useful feature for commute/biking recommendations
- **Subway ridership peaks 8–9am and 5–6pm** — morning/evening commute windows
- **Crashes peak 4–6pm** — rush hour driving risk, important safety signal
- **Brooklyn has the most crime by volume** but Manhattan has highest density per sq mile
- **Risk score confirms 4–6pm is the most active window** across all signals simultaneously

These patterns directly informed the feature engineering choices — rolling 24h windows, hour/day_of_week features, and the 75th percentile thresholds for high vs low classification.